General

In [ ]:
import json
import os
import re
import pandas as pd

# Paths and Model Configuration
base_path_origin = "PATH1"
base_path_hallucination = "PATH2"
model_name = "MODEL-NAME" 

# Dataset Mapping: Original Performance
origin_map = {
    "CT_test.jsonl": "CT",
    "MRI_test.jsonl": "MRI",
    "X-Ray_test.jsonl": "X-Ray",
    "ultrasound_test.jsonl": "US",
    "Dermoscopy_test.jsonl": "Der",
    "Fundus_test.jsonl": "FP",
    "OCT_test.jsonl": "OCT",
    "Microscopy_test.jsonl": "Micro"
}

# Dataset Mapping: Hallucination Categories (Ordered as requested: CMI, NOI, CCA, PI)
hallu_map = {
    "Cross-Modal_Incongruity.jsonl": "CMI",
    "Negative-Option_Induction.jsonl": "NOI",
    "Counterfactual_Clinical_Assumptions.jsonl": "CCA",
    "Perceptual_Insufficiency.jsonl": "PI"
}

def calculate_accuracy(file_path):
    """
    Parses JSONL files and calculates the accuracy based on specific 
    regex patterns for ground truth and model predictions.
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        return 0.0
    
    total = 0
    correct = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line: continue
                item = json.loads(line)
                gt_raw = item.get('gt', '')
                pred_raw = item.get('model_output', '')
                
                # Extract Ground Truth
                gt_match = re.search(r'<answer>\s*([a-zA-Z])\s*</answer>', gt_raw)
                if gt_match:
                    gt_label = gt_match.group(1).upper()
                else:
                    continue 
                
                pred_clean = pred_raw.strip()
                pred_label = ""
                
                # Prediction Parsing Logic
                if re.fullmatch(r'[a-eA-E]', pred_clean):
                    pred_label = pred_clean.upper()
                elif re.search(r'([a-eA-E])<\|endoftext\|>', pred_raw):
                    match = re.search(r'([a-eA-E])<\|endoftext\|>', pred_raw)
                    pred_label = match.group(1).upper()
                elif re.search(r'^\s*([a-eA-E])(?:[\)\s].*?)?<\|end\|>', pred_raw, re.DOTALL):
                    match = re.search(r'^\s*([a-eA-E])(?:[\)\s].*?)?<\|end\|>', pred_raw, re.DOTALL)
                    pred_label = match.group(1).upper()
                else:
                    matches = re.findall(r'([a-eA-E])\)', pred_raw)
                    if len(matches) == 1:
                        pred_label = matches[0].upper()
                
                if pred_label and gt_label == pred_label:
                    correct += 1
                total += 1
            except (json.JSONDecodeError, Exception):
                continue
                
    return correct / total if total > 0 else 0.0

# Initialize result dictionary
results = {"Model Name": model_name}

# 1. Process Origin Results
origin_scores = []
print(f"Processing Model: {model_name}")
print("--- Processing Origin Results ---")
for filename, col_name in origin_map.items():
    full_path = os.path.join(base_path_origin, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    origin_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_origin = sum(origin_scores) / len(origin_scores) if origin_scores else 0
results["Avg_Origin"] = avg_origin 

# 2. Process Hallucination Results
hallu_scores = []
print("\n--- Processing Hallucination Results ---")
for filename, col_name in hallu_map.items():
    full_path = os.path.join(base_path_hallucination, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    hallu_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_hallu = sum(hallu_scores) / len(hallu_scores) if hallu_scores else 0
results["Avg_Hallu"] = avg_hallu

avg_origin_rounded = round(avg_origin, 3)
avg_hallu_rounded = round(avg_hallu, 3)
# 3. Calculate OMHScore (Harmonic Mean)
if (avg_origin_rounded + avg_hallu_rounded) > 0:
    omh_score = (2 * avg_origin_rounded * avg_hallu_rounded) / (avg_origin_rounded + avg_hallu_rounded)
else:
    omh_score = 0.0
results["OMHScore"] = omh_score

# Define Column Order (Updated for CMI, NOI, CCA, PI and OMHScore)
columns_order = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg_Origin",
    "CMI", "NOI", "CCA", "PI", "Avg_Hallu", "OMHScore"
]

# Display Headers
final_headers = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg.",
    "CMI", "NOI", "CCA", "PI", "Avg.", "OMHScore"
]

# Create DataFrame
df = pd.DataFrame([results])
df = df[columns_order] 

# Formatting
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
df[numeric_cols] = df[numeric_cols].round(3)

# Save to CSV
output_file = "./evaluation_results.csv"
file_exists = os.path.exists(output_file)

if not file_exists:
    df.columns = final_headers
    df.to_csv(output_file, index=False, mode='w', encoding='utf-8-sig')
    print(f"\n[New File] Saved to: {os.path.abspath(output_file)}")
else:
    df.to_csv(output_file, index=False, mode='a', header=False, encoding='utf-8-sig')
    print(f"\n[Append] Updated in: {os.path.abspath(output_file)}")

print("-" * 80)
df_display = df.copy()
if len(df_display.columns) == len(final_headers):
    df_display.columns = final_headers
print(df_display.to_string(index=False))

InternVL3.5-8B

In [ ]:
import json
import os
import re
import pandas as pd

# ================= Configuration & Paths =================
base_path_origin = "PATH1"
base_path_hallucination = "PATH2"

model_name = "InternVL3_5-8B-HF" 

# Mapping for Original Performance datasets
origin_map = {
    "CT(Computed Tomography)_test.jsonl": "CT",
    "MR (Mag-netic Resonance Imaging)_test.jsonl": "MRI",
    "X-Ray_test.jsonl": "X-Ray",
    "ultrasound_test.jsonl": "US",
    "Dermoscopy_test.jsonl": "Der",
    "Fundus Photography_test.jsonl": "FP",
    "OCT (Optical Coherence Tomography_test.jsonl": "OCT",
    "Microscopy Images_test.jsonl": "Micro"
}

# Mapping for Hallucination datasets (Updated names and order: CMI, NOI, CCA, PI)
hallu_map = {
    "Cross-Modal_Incongruity.jsonl": "CMI",
    "Negative-Option_Induction.jsonl": "NOI",
    "Counterfactual_Clinical_Assumptions.jsonl": "CCA",
    "Perceptual_Insufficiency.jsonl": "PI"
}

# ================= Core Processing Function =================

def calculate_accuracy(file_path):
    """Parses JSONL and calculates accuracy with multi-step fallback logic."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        return 0.0
    
    total = 0
    correct = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line: continue
                item = json.loads(line)
                gt_raw = item.get('gt', '')
                pred_raw = item.get('model_output', '')
                
                # 1. Extract Ground Truth (<answer> A </answer>)
                gt_match = re.search(r'<answer>\s*([a-zA-Z])\s*</answer>', gt_raw)
                if gt_match:
                    gt_label = gt_match.group(1).upper()
                else:
                    continue 
                
                # 2. Extract Model Output
                pred_clean = pred_raw.strip()
                pred_label = ""

                # Case A: Exact match (single letter A-E)
                if re.fullmatch(r'[a-eA-E]', pred_clean):
                    pred_label = pred_clean.upper()
                
                else:
                    # Case B: Extract last character (handles "\n\nC" or "Answer: C")
                    if len(pred_clean) > 0:
                        last_char = pred_clean[-1].upper()
                        if last_char in ['A', 'B', 'C', 'D', 'E']:
                            pred_label = last_char

                    # Case C: Fallback to "A)" pattern
                    if not pred_label:
                        matches = re.findall(r'([a-eA-E])\)', pred_raw)
                        if len(matches) == 1:
                            pred_label = matches[0].upper()
                
                # 3. Comparison
                if pred_label and gt_label == pred_label:
                    correct += 1
                total += 1
                
            except (json.JSONDecodeError, Exception):
                continue
                
    return correct / total if total > 0 else 0.0

# ================= Execution Logic =================

results = {"Model Name": model_name}

# 1. Process Origin Results
origin_scores = []
print(f"Processing Model: {model_name}")
print("--- Processing Origin Results ---")
for filename, col_name in origin_map.items():
    full_path = os.path.join(base_path_origin, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    origin_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_origin = sum(origin_scores) / len(origin_scores) if origin_scores else 0
results["Avg_Origin"] = avg_origin 

# 2. Process Hallucination Results
hallu_scores = []
print("\n--- Processing Hallucination Results ---")
for filename, col_name in hallu_map.items():
    full_path = os.path.join(base_path_hallucination, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    hallu_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_hallu = sum(hallu_scores) / len(hallu_scores) if hallu_scores else 0
results["Avg_Hallu"] = avg_hallu

avg_origin_rounded = round(avg_origin, 3)
avg_hallu_rounded = round(avg_hallu, 3)
# 3. Calculate OMHScore (Harmonic Mean)
if (avg_origin_rounded + avg_hallu_rounded) > 0:
    omh_score = (2 * avg_origin_rounded * avg_hallu_rounded) / (avg_origin_rounded + avg_hallu_rounded)
else:
    omh_score = 0.0
results["OMHScore"] = omh_score

# ================= DataFrame Construction & Export =================

# Internal column order for data alignment
columns_order = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg_Origin",
    "CMI", "NOI", "CCA", "PI", "Avg_Hallu", "OMHScore"
]

# Display headers for the CSV and print output
final_headers = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg.",
    "CMI", "NOI", "CCA", "PI", "Avg.", "OMHScore"
]

df = pd.DataFrame([results])
df = df[columns_order] 

# Formatting numerical values to 3 decimal places
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
df[numeric_cols] = df[numeric_cols].round(3)

# Save to CSV (Append mode)
output_file = "./evaluation_results.csv"
file_exists = os.path.exists(output_file)

if not file_exists:
    df.columns = final_headers
    df.to_csv(output_file, index=False, mode='w', encoding='utf-8-sig')
    print(f"\n[New File Created] Results saved to: {os.path.abspath(output_file)}")
else:
    df.to_csv(output_file, index=False, mode='a', header=False, encoding='utf-8-sig')
    print(f"\n[Data Appended] Results updated in: {os.path.abspath(output_file)}")

print("-" * 85)
# Display formatted table
df_display = df.copy()
if len(df_display.columns) == len(final_headers):
    df_display.columns = final_headers
print(df_display.to_string(index=False))


BiMediX2-8B

In [ ]:
import json
import os
import re
import pandas as pd

# ================= Configuration & Paths =================
base_path_origin = "PATH1"
base_path_hallucination = "PATH2"

model_name = "BiMediX2-8B"

# Mapping for Original Performance datasets (BiMediX2 specific filenames)
origin_map = {
    "CT(Computed Tomography)_test.jsonl": "CT",
    "MR (Mag-netic Resonance Imaging)_test.jsonl": "MRI",
    "X-Ray_test.jsonl": "X-Ray",
    "ultrasound_test.jsonl": "US",
    "Dermoscopy_test.jsonl": "Der",
    "Fundus Photography_test.jsonl": "FP",
    "OCT (Optical Coherence Tomography_test.jsonl": "OCT",
    "Microscopy Images_test.jsonl": "Micro"
}

# Mapping for Hallucination datasets (Updated names and order: CMI, NOI, CCA, PI)
hallu_map = {
    "Cross-Modal_Incongruity.jsonl": "CMI",
    "Negative-Option_Induction.jsonl": "NOI",
    "Counterfactual_Clinical_Assumptions.jsonl": "CCA",
    "Perceptual_Insufficiency.jsonl": "PI"
}

# ================= Core Processing Functions =================

def extract_text_by_label(question, label):
    """
    Extracts the text content associated with a specific label (A/B/C/D) from the question.
    Example: "... A)Ureter, B)aorta, C)Thyroid Gland ..."
    Extracting B -> returns "aorta"
    """
    if not question:
        return None
    
    # Pattern: Label + ) + content (until next comma or end of string)
    pattern = re.escape(label) + r'\)\s*([^,]+)'
    match = re.search(pattern, question, re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    return None

def calculate_accuracy(file_path):
    """
    Accuracy Logic:
    1. First, try to match the Ground Truth letter (A-E) in the output.
    2. If no letter match, extract the text description of the GT option 
       from the question and check if that text exists in the output.
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        return 0.0
    
    total = 0
    correct = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line: continue
                item = json.loads(line)
                
                gt_raw = item.get('gt', '')
                pred_raw = item.get('model_output', '')
                question_text = item.get('question', '')
                
                # 1. Extract Ground Truth Letter
                gt_match = re.search(r'<answer>\s*([a-zA-Z])\s*</answer>', gt_raw)
                if gt_match:
                    gt_label = gt_match.group(1).upper()
                else:
                    continue 
                
                is_hit = False
                
                # --- Step A: Letter Matching (Priority) ---
                first_letter_match = re.search(r'[a-zA-Z]', pred_raw)
                if first_letter_match and first_letter_match.group(0).upper() == gt_label:
                    is_hit = True
                
                # --- Step B: Text Content Matching (Fallback) ---
                if not is_hit:
                    option_text = extract_text_by_label(question_text, gt_label)
                    # Match description if it's long enough to be meaningful
                    if option_text and len(option_text) > 1:
                        if option_text.lower() in pred_raw.lower():
                            is_hit = True
                
                if is_hit:
                    correct += 1
                total += 1
                
            except Exception:
                continue
                
    return correct / total if total > 0 else 0.0

# ================= Execution Logic =================

results = {"Model Name": model_name}

# 1. Process Origin Results
origin_scores = []
print(f"Processing Model: {model_name}")
print("--- Processing Origin Results ---")
for filename, col_name in origin_map.items():
    acc = calculate_accuracy(os.path.join(base_path_origin, filename))
    results[col_name] = acc
    origin_scores.append(acc)
    print(f"  Origin {col_name}: {acc:.3f}")

avg_origin = sum(origin_scores) / len(origin_scores) if origin_scores else 0
results["Avg_Origin"] = avg_origin 

# 2. Process Hallucination Results
hallu_scores = []
print("\n--- Processing Hallucination Results ---")
for filename, col_name in hallu_map.items():
    acc = calculate_accuracy(os.path.join(base_path_hallucination, filename))
    results[col_name] = acc
    hallu_scores.append(acc)
    print(f"  Hallu {col_name}: {acc:.3f}")

avg_hallu = sum(hallu_scores) / len(hallu_scores) if hallu_scores else 0
results["Avg_Hallu"] = avg_hallu

avg_origin_rounded = round(avg_origin, 3)
avg_hallu_rounded = round(avg_hallu, 3)
# 3. Calculate OMHScore (Harmonic Mean)
if (avg_origin_rounded + avg_hallu_rounded) > 0:
    omh_score = (2 * avg_origin_rounded * avg_hallu_rounded) / (avg_origin_rounded + avg_hallu_rounded)
else:
    omh_score = 0.0
results["OMHScore"] = omh_score

# ================= DataFrame Construction & Export =================

# Updated order: Origin metrics -> CMI -> NOI -> CCA -> PI -> Avg_Hallu -> OMHScore
columns_order = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg_Origin",
    "CMI", "NOI", "CCA", "PI", "Avg_Hallu", "OMHScore"
]

final_headers = [
    "Model Name", 
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg.",
    "CMI", "NOI", "CCA", "PI", "Avg.", "OMHScore"
]

df = pd.DataFrame([results])
df = df[columns_order]

# Round numeric values to 3 decimal places
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
df[numeric_cols] = df[numeric_cols].round(3)

# Save to CSV
output_file = "./evaluation_results.csv"
file_exists = os.path.exists(output_file)

if not file_exists:
    df.columns = final_headers
    df.to_csv(output_file, index=False, mode='w', encoding='utf-8-sig')
    print(f"\n[New File] Results saved to: {os.path.abspath(output_file)}")
else:
    df.to_csv(output_file, index=False, mode='a', header=False, encoding='utf-8-sig')
    print(f"\n[Append] Results updated in: {os.path.abspath(output_file)}")

print("-" * 85)
# Re-apply headers for terminal display
df_display = df.copy()
if len(df_display.columns) == len(final_headers):
    df_display.columns = final_headers
print(df_display.to_string(index=False))


Janus-Pro

In [ ]:
import json
import os
import re
import pandas as pd

# ================= Configuration & Paths =================
base_path_origin = "PATH1"
base_path_hallucination = "PATH2"

model_name = "Janus-Pro-7B"

# Mapping for Original Performance datasets
origin_map = {
    "CT_test.jsonl": "CT",
    "MRI_test.jsonl": "MRI",
    "X-Ray_test.jsonl": "X-Ray",
    "ultrasound_test.jsonl": "US",
    "Dermoscopy_test.jsonl": "Der",
    "Fundus_test.jsonl": "FP",
    "OCT_test.jsonl": "OCT",
    "Microscopy_test.jsonl": "Micro"
}

# Mapping for Hallucination datasets (Updated names and order: CMI, NOI, CCA, PI)
hallu_map = {
    "Cross-Modal_Incongruity.jsonl": "CMI",
    "Negative-Option_Induction.jsonl": "NOI",
    "Counterfactual_Clinical_Assumptions.jsonl": "CCA",
    "Perceptual_Insufficiency.jsonl": "PI"
}

# ================= Core Processing Function =================

def calculate_accuracy(file_path):
    """Parses JSONL and calculates accuracy by extracting the last standalone letter."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        return 0.0
    
    total = 0
    correct = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line:
                    continue

                item = json.loads(line)
                gt_raw = item.get('gt', '')
                pred_raw = item.get('model_output', '')
                
                # 1. Extract Ground Truth (<answer> A </answer>)
                gt_match = re.search(r'<answer>\s*([a-zA-Z])\s*</answer>', gt_raw)
                if gt_match:
                    gt_label = gt_match.group(1).upper()
                else:
                    continue

                
                #Janus-Pro-1B
                # pred_clean = pred_raw.strip()
                # pred_label = ""
                # if re.fullmatch(r'[a-eA-E]', pred_clean):
                #     pred_label = pred_clean.upper()
                # else:
                #     match = re.search(r'Answer:\s*([a-eA-E])', pred_raw, re.IGNORECASE)
                #     if match:
                #         pred_label = match.group(1).upper()
                

                # 2. Extract Model Output
                # Logic: Find all standalone letters (A-E) and take the last one
                pred_label = ""
                matches = re.findall(r'\b([a-eA-E])\b', pred_raw)
                if matches:
                    pred_label = matches[-1].upper()

                # 3. Comparison
                if pred_label and gt_label == pred_label:
                    correct += 1
                total += 1
                
            except (json.JSONDecodeError, Exception):
                continue
                
    return correct / total if total > 0 else 0.0


# ================= Execution Logic =================

results = {"Model Name": model_name}

# 1. Process Origin Results
origin_scores = []
print(f"Processing Model: {model_name}")
print("--- Processing Origin Results ---")
for filename, col_name in origin_map.items():
    full_path = os.path.join(base_path_origin, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    origin_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_origin = sum(origin_scores) / len(origin_scores) if origin_scores else 0
results["Avg_Origin"] = avg_origin

# 2. Process Hallucination Results
hallu_scores = []
print("\n--- Processing Hallucination Results ---")
for filename, col_name in hallu_map.items():
    full_path = os.path.join(base_path_hallucination, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    hallu_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_hallu = sum(hallu_scores) / len(hallu_scores) if hallu_scores else 0
results["Avg_Hallu"] = avg_hallu

avg_origin_rounded = round(avg_origin, 3)
avg_hallu_rounded = round(avg_hallu, 3)
# 3. Calculate OMHScore (Harmonic Mean)
if (avg_origin_rounded + avg_hallu_rounded) > 0:
    omh_score = (2 * avg_origin_rounded * avg_hallu_rounded) / (avg_origin_rounded + avg_hallu_rounded)
else:
    omh_score = 0.0
results["OMHScore"] = omh_score

# ================= DataFrame Construction & Export =================

# Internal column order for data alignment
columns_order = [
    "Model Name",
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg_Origin",
    "CMI", "NOI", "CCA", "PI", "Avg_Hallu", "OMHScore"
]

# Display headers for the CSV and print output
final_headers = [
    "Model Name",
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg.",
    "CMI", "NOI", "CCA", "PI", "Avg.", "OMHScore"
]

df = pd.DataFrame([results])
df = df[columns_order]

# Format numerical columns to 3 decimal places
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
df[numeric_cols] = df[numeric_cols].round(3)

# Save to CSV (Append mode)
output_file = "./evaluation_results.csv"
file_exists = os.path.exists(output_file)

if not file_exists:
    df.columns = final_headers
    df.to_csv(output_file, index=False, mode='w', encoding='utf-8-sig')
    print(f"\n[New File] Results saved to: {os.path.abspath(output_file)}")
else:
    df.to_csv(output_file, index=False, mode='a', header=False, encoding='utf-8-sig')
    print(f"\n[Data Appended] Results updated in: {os.path.abspath(output_file)}")

print("-" * 85)
# Display formatted table
df_display = df.copy()
if len(df_display.columns) == len(final_headers):
    df_display.columns = final_headers
print(df_display.to_string(index=False))


ChexAgent

In [ ]:
import json
import os
import re
import pandas as pd

# ================= Configuration & Paths =================
base_path_origin = "PATH1"
base_path_hallucination = "PATH2"

model_name = "CheXagent"

# Mapping for Original Performance datasets
origin_map = {
    "CT_test.jsonl": "CT",
    "MRI_test.jsonl": "MRI",
    "X-Ray_test.jsonl": "X-Ray",
    "ultrasound_test.jsonl": "US",
    "Dermoscopy_test.jsonl": "Der",
    "Fundus_test.jsonl": "FP",
    "OCT_test.jsonl": "OCT",
    "Microscopy_test.jsonl": "Micro"
}

# Mapping for Hallucination datasets (Updated names and order: CMI, NOI, CCA, PI)
hallu_map = {
    "Cross-Modal_Incongruity.jsonl": "CMI",
    "Negative-Option_Induction.jsonl": "NOI",
    "Counterfactual_Clinical_Assumptions.jsonl": "CCA",
    "Perceptual_Insufficiency.jsonl": "PI"
}

# ================= Core Processing Function =================

def calculate_accuracy(file_path):
    """Parses JSONL and calculates accuracy based on unique occurrences of A)-E)."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found: {file_path}")
        return 0.0
    
    total = 0
    correct = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                line = line.strip()
                if not line:
                    continue

                item = json.loads(line)
                gt_raw = item.get('gt', '')
                pred_raw = item.get('model_output', '')
                
                # 1. Extract Ground Truth (<answer> A </answer>)
                gt_match = re.search(r'<answer>\s*([a-zA-Z])\s*</answer>', gt_raw)
                if gt_match:
                    gt_label = gt_match.group(1).upper()
                else:
                    continue
                
                # 2. Extract Model Output
                # CheXagent Logic: Check for a unique occurrence of the "X)" pattern
                pred_label = ""
                matches = re.findall(r'([A-E])\)', pred_raw)

                if len(matches) == 1:
                    pred_label = matches[0].upper()
                else:
                    pred_label = ""

                # 3. Comparison
                if pred_label and gt_label == pred_label:
                    correct += 1
                total += 1
                
            except (json.JSONDecodeError, Exception):
                continue
                
    return correct / total if total > 0 else 0.0


# ================= Execution Logic =================

results = {"Model Name": model_name}

# 1. Process Origin Results
origin_scores = []
print(f"Processing Model: {model_name}")
print("--- Processing Origin Results ---")
for filename, col_name in origin_map.items():
    full_path = os.path.join(base_path_origin, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    origin_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_origin = sum(origin_scores) / len(origin_scores) if origin_scores else 0
results["Avg_Origin"] = avg_origin

# 2. Process Hallucination Results
hallu_scores = []
print("\n--- Processing Hallucination Results ---")
for filename, col_name in hallu_map.items():
    full_path = os.path.join(base_path_hallucination, filename)
    acc = calculate_accuracy(full_path)
    results[col_name] = acc
    hallu_scores.append(acc)
    print(f"  {col_name}: {acc:.3f}")

avg_hallu = sum(hallu_scores) / len(hallu_scores) if hallu_scores else 0
results["Avg_Hallu"] = avg_hallu

avg_origin_rounded = round(avg_origin, 3)
avg_hallu_rounded = round(avg_hallu, 3)
# 3. Calculate OMHScore (Harmonic Mean)
if (avg_origin_rounded + avg_hallu_rounded) > 0:
    omh_score = (2 * avg_origin_rounded * avg_hallu_rounded) / (avg_origin_rounded + avg_hallu_rounded)
else:
    omh_score = 0.0
results["OMHScore"] = omh_score

# ================= DataFrame Construction & Export =================

# Internal column order for data alignment
columns_order = [
    "Model Name",
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg_Origin",
    "CMI", "NOI", "CCA", "PI", "Avg_Hallu", "OMHScore"
]

# Display headers for the CSV and terminal output
final_headers = [
    "Model Name",
    "CT", "MRI", "X-Ray", "US", "Der", "FP", "OCT", "Micro", "Avg.",
    "CMI", "NOI", "CCA", "PI", "Avg.", "OMHScore"
]

df = pd.DataFrame([results])
df = df[columns_order]

# Format numerical columns to 3 decimal places
numeric_cols = df.select_dtypes(include=['float', 'int']).columns
df[numeric_cols] = df[numeric_cols].round(3)

# Save to CSV (Append mode)
output_file = "./evaluation_results.csv"
file_exists = os.path.exists(output_file)

if not file_exists:
    df.columns = final_headers
    df.to_csv(output_file, index=False, mode='w', encoding='utf-8-sig')
    print(f"\n[New File] Results saved to: {os.path.abspath(output_file)}")
else:
    df.to_csv(output_file, index=False, mode='a', header=False, encoding='utf-8-sig')
    print(f"\n[Append] Results updated in: {os.path.abspath(output_file)}")

print("-" * 85)
# Display formatted table
df_display = df.copy()
if len(df_display.columns) == len(final_headers):
    df_display.columns = final_headers
print(df_display.to_string(index=False))
